# PyBaMM SEI Formulations Sensitivity Analysis
**Last updated: 2026-04-03 11:47 (Local Time)**
This notebook simulates calendar ageing followed by fast cycling, designed to run one model at a time.

In [ ]:
!pip install pybamm==26.3 pandas matplotlib

In [ ]:
import pybamm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pybamm.set_logging_level("INFO")

## Base Configuration
**Select your SEI mechanism below.** To test all four, run the entire notebook, save your CSV data, change the string here, and run the notebook again!

Available options:
* `"reaction limited"`
* `"solvent-diffusion limited"`
* `"electron-migration limited"`
* `"interstitial-diffusion limited"`

In [ ]:
CURRENT_MODEL = "solvent-diffusion limited"  # <--- Change this string!

STORAGE_DAYS = 30
INITIAL_SOC = 0.9
TOTAL_CYCLES = 100
CHUNK_SIZE = 10

EXPERIMENT_STEP = (
    "Discharge at C/9 until 3.2 V",
    "Rest for 15 minutes",
    "Charge at C/2 until 4.1 V",
    "Hold at 4.1 V until C/20",
    "Rest for 15 minutes",
)

def get_submesh_types():
    model = pybamm.lithium_ion.DFN()
    return model.default_submesh_types.copy()

var_pts = {"x_n": 30, "x_s": 30, "x_p": 30, "r_n": 50, "r_p": 50}

## Main Execution Framework
Executes 30 days of storage followed by exactly `TOTAL_CYCLES` divided firmly into memory-safe chunks.

In [ ]:
print(f"\n{'='*50}")
print(f"Running Isolated SEI Model: {CURRENT_MODEL}")
print(f"{'='*50}\n")

options = {
    "SEI": CURRENT_MODEL,
    "SEI porosity change": "true",
    "lithium plating": "partially reversible",
    "lithium plating porosity change": "true",
    "particle mechanics": ("swelling and cracking", "swelling only"),
    "SEI on cracks": "true",
    "loss of active material": "stress-driven",
}

model = pybamm.lithium_ion.DFN(options)
parameter_values = pybamm.ParameterValues("OKane2022")
parameter_values["Current function [A]"] = 0
solver = pybamm.IDAKLUSolver(atol=1e-6, rtol=1e-6)

# Ageing
print(f"-> Storage Phase ({STORAGE_DAYS} days)")
seconds = STORAGE_DAYS * 24 * 60 * 60
t_eval = np.linspace(0, seconds, min(100, max(20, STORAGE_DAYS * 2)))
sim = pybamm.Simulation(
    model, parameter_values=parameter_values, solver=solver, var_pts=var_pts, submesh_types=get_submesh_types()
)
sol_ageing = sim.solve(t_eval=t_eval, initial_soc=INITIAL_SOC)

# Cycling
print(f"-> Cycling Phase ({TOTAL_CYCLES} cycles)")
num_chunks = TOTAL_CYCLES // CHUNK_SIZE
experiment_chunk = pybamm.Experiment([EXPERIMENT_STEP] * CHUNK_SIZE)

cc_caps, cv_caps, dis_caps, cycles = [], [], [], []
current_solution = sol_ageing

parameter_values_cyc = pybamm.ParameterValues("OKane2022")

for chunk_idx in range(num_chunks):
    print(f"   Chunk {chunk_idx + 1}/{num_chunks}... ", end="")
    sim_cyc = pybamm.Simulation(
        model,
        experiment=experiment_chunk,
        parameter_values=parameter_values_cyc,
        solver=solver,
        var_pts=var_pts,
        submesh_types=get_submesh_types(),
    )
    
    chunk_solution = sim_cyc.solve(starting_solution=current_solution)
    
    # Parse data
    for cycle_sol in chunk_solution.cycles:
        steps = cycle_sol.steps
        step_dis = steps[0] if len(steps) > 0 else None
        charge_c_step = steps[2] if len(steps) > 2 else None
        charge_v_step = steps[3] if len(steps) > 3 else None
        
        d_cap = (
            step_dis["Discharge capacity [A.h]"].entries[-1]
            - step_dis["Discharge capacity [A.h]"].entries[0]
        ) if step_dis else 0.0
        
        c_cap = abs(
            charge_c_step["Discharge capacity [A.h]"].entries[-1]
            - charge_c_step["Discharge capacity [A.h]"].entries[0]
        ) if charge_c_step else 0.0
        
        v_cap = abs(
            charge_v_step["Discharge capacity [A.h]"].entries[-1]
            - charge_v_step["Discharge capacity [A.h]"].entries[0]
        ) if charge_v_step else 0.0
        
        dis_caps.append(d_cap)
        cc_caps.append(c_cap)
        cv_caps.append(v_cap)
        cycles.append(len(cycles) + 1)
        
    current_solution = chunk_solution
    print("Done.")

## Analysis and Data Export
This outputs the CC/CV table specifically for your chosen isolated model and writes it dynamically to a CSV file!

In [ ]:
rows = []
for c, cc, cv in zip(cycles, cc_caps, cv_caps):
    rows.append({
        "Variant": CURRENT_MODEL,
        "Cycle": c,
        "CC Capacity [A.h]": cc,
        "CV Capacity [A.h]": cv,
        "CC/CV Ratio": cc / cv if cv > 0 else float('nan')
    })
    
df = pd.DataFrame(rows)

milestones = [10, 50, 100, 150, 200]
df_milestones = df[df['Cycle'].isin(milestones)]
print(f"\n--- {CURRENT_MODEL} Analysis Table ---")
print(df_milestones.to_string(index=False))

filename = f"{CURRENT_MODEL.replace(' ', '_').replace('-', '_')}_data.csv"
df.to_csv(filename, index=False)
print(f"\nSaved raw cycle data to: {filename}")

## Visual Plots
Live rendering of the isolated execution.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 6))

axs[0].plot(cycles, cc_caps, marker=".", label=CURRENT_MODEL, color='blue')
axs[0].set_title("CC Capacity Decay")
axs[0].set_ylabel("Capacity [A.h]")
axs[0].set_xlabel("Cycles")
axs[0].grid(True)
axs[0].legend()

axs[1].plot(cycles, cv_caps, marker=".", label=CURRENT_MODEL, color='orange')
axs[1].set_title("CV Capacity Reliance")
axs[1].set_ylabel("Capacity [A.h]")
axs[1].set_xlabel("Cycles")
axs[1].grid(True)
axs[1].legend()

ratio = [cc / cv if cv > 0 else 0 for cc, cv in zip(cc_caps, cv_caps)]
axs[2].plot(cycles, ratio, marker=".", label=CURRENT_MODEL, color='green')
axs[2].set_title("CC/CV Ratio")
axs[2].set_ylabel("Ratio")
axs[2].set_xlabel("Cycles")
axs[2].grid(True)
axs[2].legend()

plt.tight_layout()
plt.show()